# Prestações de Contas — Condomínio Humaitá

Ingestão e limpeza das prestações de contas mensais exportadas do sistema **ClienteOnline** (imobiliária).

- **Fonte**: `exports/hojas/prestacao/` — 15 arquivos `.xlsx` (mai/2022 – ago/2023)
- **Formato**: XLSX real com 3 abas: `Receitas`, `Despesas`, `Resumo por Subconta`
- **Saída**: `exports/csv/prestacoes.csv`

> **Nota**: gap de dados ago/2023–ago/2025 (PDFs disponíveis mas fora do escopo desta etapa).

In [1]:
%pip install -q pandas openpyxl matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
from pathlib import Path

import pandas as pd

In [ ]:
def parse_br_number(s):
    """Converte string no formato brasileiro (1.234,56) para float."""
    if isinstance(s, (int, float)):
        return float(s) if not pd.isna(s) else None
    if not isinstance(s, str):
        return None
    s = s.strip()
    if s in ("-", "", "nan", "None"):
        return None
    s = re.sub(r"\s", "", s)
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return None


def _is_skip_row(evento_str: str) -> bool:
    """Retorna True para linhas que não são lançamentos individuais."""
    s = str(evento_str).strip()
    if not s or s in ("nan", "None"):
        return True
    # Cabeçalhos de grupo com ** (ex: "** Pessoal/Encargos/Adm **", "** Água/Luz/Gás **")
    if s.startswith("**") or s.endswith("**"):
        return True
    lower = s.lower()
    # Subtotais, totais e títulos de aba
    if any(lower.startswith(p) for p in ("subtotal", "total", "receitas", "despesas")):
        return True
    # "Outros Eventos" é subtotal de grupo (soma dos itens do bloco acima)
    if lower.startswith("outros eventos"):
        return True
    return False


def parse_prestacao_file(filepath: Path) -> pd.DataFrame:
    """
    Parseia um arquivo .xlsx de Prestação de Contas.

    Estrutura esperada (por aba):
      Linha 0 : título da aba (ex: "Receitas")
      Linha 1 : cabeçalho ("Evento", "Valor")
      Linhas 2+ : dados + linhas de subtotal (a serem ignoradas)

    Retorna DataFrame com: mes_ano, tipo, evento, valor.
    """
    parts = filepath.stem.split("_")
    mes_ano = f"{parts[0]}-{parts[1].zfill(2)}"

    xl = pd.ExcelFile(filepath)
    frames = []

    for sheet, tipo in [("Receitas", "receita"), ("Despesas", "despesa")]:
        if sheet not in xl.sheet_names:
            continue

        raw = xl.parse(sheet, header=None)

        # Linha 1 = cabeçalho; dados a partir da linha 2
        df = raw.iloc[2:].copy()
        df.columns = list(range(df.shape[1]))
        df = df[[0, 1]].copy()
        df.columns = ["evento", "valor"]

        # Remover subtotais, cabeçalhos de grupo e linhas vazias
        df = df[~df["evento"].apply(_is_skip_row)].copy()
        df = df.dropna(subset=["evento"]).copy()

        df["valor"]   = df["valor"].apply(parse_br_number)
        df["tipo"]    = tipo
        df["mes_ano"] = mes_ano

        frames.append(df[["mes_ano", "tipo", "evento", "valor"]])

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [6]:
PRESTACAO_DIR = Path("../exports/hojas/prestacao")
files = sorted(PRESTACAO_DIR.glob("*.xlsx"))
print(f"Arquivos encontrados: {len(files)}\n")

dfs = []
for f in files:
    df = parse_prestacao_file(f)
    print(f"  {f.name}: {len(df)} registros")
    dfs.append(df)

df_prestacoes = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(df_prestacoes)} registros")
df_prestacoes.head(10)

Arquivos encontrados: 15

  2022_05.xlsx: 36 registros
  2022_06.xlsx: 44 registros
  2022_07.xlsx: 50 registros
  2022_08.xlsx: 51 registros
  2022_09.xlsx: 49 registros
  2022_10.xlsx: 46 registros
  2022_11.xlsx: 44 registros
  2022_12.xlsx: 52 registros
  2023_01.xlsx: 48 registros
  2023_02.xlsx: 55 registros
  2023_03.xlsx: 52 registros
  2023_04.xlsx: 51 registros
  2023_05.xlsx: 47 registros
  2023_07.xlsx: 50 registros
  2023_08.xlsx: 53 registros

Total: 728 registros


,mes_ano,tipo,evento,valor
0,2022-05,receita,Outros Eventos,43389.08
1,2022-05,receita,Rec.condomínio,41248.84
2,2022-05,receita,Fundo Reserva,1927.26
3,2022-05,receita,Rec.taxa Uso Box,190.00
4,2022-05,receita,Fundo Obras,22.98
5,2022-05,receita,Rec. Multa,108.19
6,2022-05,receita,Rec.multa+C.m.+Jrs.,23.31
7,2022-05,despesa,Pg.salário,4637.14
8,2022-05,despesa,Pg.fgts,268.34
9,2022-05,despesa,Pg.adto.salário,600.00


In [7]:
# Normalização da coluna evento: uppercase, strip, limpar espaços duplos
df_prestacoes["evento"] = (
    df_prestacoes["evento"]
    .astype(str)
    .str.strip()
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
)

print("=== Tipos ===")
print(df_prestacoes["tipo"].value_counts().to_string())

print("\n=== Período ===")
print(df_prestacoes["mes_ano"].unique())

print("\n=== Top 30 Eventos ===")
print(df_prestacoes["evento"].value_counts().head(30).to_string())

print("\n=== Estatísticas de valor por tipo ===")
print(df_prestacoes.groupby("tipo")["valor"].describe().to_string())

=== Tipos ===
tipo
despesa    576
receita    152

=== Período ===
<StringArray>
['2022-05', '2022-06', '2022-07', '2022-08', '2022-09', '2022-10', '2022-11',
 '2022-12', '2023-01', '2023-02', '2023-03', '2023-04', '2023-05', '2023-07',
 '2023-08']
Length: 15, dtype: str

=== Top 30 Eventos ===
evento
REC.CONDOMÍNIO                  18
OUTROS EVENTOS                  17
FUNDO OBRAS                     15
REC. MULTA                      15
REC.MULTA+C.M.+JRS.             15
PG.SALÁRIO                      15
PG.FGTS                         15
PG.ADTO.SALÁRIO                 15
PG.VALE REFEIÇÃO/ALIMENTAÇÃO    15
PG.PIS                          15
PG.DARF INSS(E-SOCIAL/REINF)    15
PG.SECOVIMED                    15
PG.VALE TRANSPORTE              15
TAXA AUXILIAR ADMINISTRACAO     15
REEMB.MAT.EXPEDIENTE            15
PG.FOLHA FUNC.BCO.              15
PG.CEEE                         15
PG.MANUT.ELEVADOR               15
PG.ISSQN                        15
PG.DECLARAÇÃO PMPA/ISSQNDEC     1

In [8]:
output_dir = Path("../exports/csv")
output_dir.mkdir(exist_ok=True)
df_prestacoes.to_csv(output_dir / "prestacoes.csv", index=False)
print(f"✓ Exportado: {len(df_prestacoes)} registros → exports/csv/prestacoes.csv")

✓ Exportado: 728 registros → exports/csv/prestacoes.csv
